# 📊 市场环境识别与可视化 v3

## 概述
- **核心算法**: TrendAnalyzer V2 (80%) + HMM V2 (20%)
- **IBD参考**: 市场跟随日/分发日辅助判断
- **回测验证**: 90.7%匹配度 (2018-2024)

---

## 1️⃣ 环境初始化

In [22]:
## 3️⃣ 14种市场状态定义\n\n**说明**: 本notebook使用的 `market_env_identifier_v3` 采用，但系统完整定义是14种状态（参考 `market_state_lib.py`）。\n\n### \n- 牛市系列(3): 强势牛市、牛市、弱势牛市\n- 熊市系列(3): 强势熊市、熊市、弱势熊市\n- 震荡系列(3): 高位震荡、中位震荡、低位震荡\n- 转折系列(2): 复苏、派发\n- 中性(1): 中性\n\n### 完整14种状态定义表\n\n# 14种市场状态定义表\nstate_14_definitions = [\n    ['牛市', '强势牛市', '全面上涨，远超年线，接近年内高点', '80%~100%', '低'],\n    ['牛市', '正常牛市', '稳健上涨，高于年线，位于较高位置', '60%~80%', '低'],\n    ['牛市', '牛市后期', '上涨动能减弱，位置较高', '40%~60%', '中'],\n    ['牛市', '牛市回调', '中期趋势向上，短期回调', '50%~70%', '中'],\n    ['熊市', '强势熊市', '全面下跌，远低于年线，接近年内低点', '0%~20%', '高'],\n    ['熊市', '正常熊市', '稳定下跌，低于年线', '10%~30%', '高'],\n    ['熊市', '熊市后期', '下跌动能减弱，位置较低', '30%~50%', '中'],\n    ['熊市', '熊市反弹', '中期趋势向下，短期反弹', '20%~40%', '中'],\n    ['震荡', '高位震荡', '在年内高位区间波动', '40%~60%', '中'],\n    ['震荡', '中位震荡', '在年内中间位置波动', '30%~50%', '中'],\n    ['震荡', '低位震荡', '在年内低位区间波动', '40%~60%', '中'],\n    ['震荡', '宽幅震荡', '大幅波动但无明确方向', '30%~50%', '高'],\n    ['转折', '底部反转', '长期下跌后开始上涨', '50%~70%', '中'],\n    ['转折', '顶部反转', '长期上涨后开始下跌', '20%~40%', '高'],\n]\n\nfig_states_14 = go.Figure(data=[go.Table(\n    header=dict(\n        values=['类别', '状态名称', '特征描述', '建议仓位', '风险等级'],\n        fill_color='#2d2d44',\n        font=dict(color='white', size=13),\n        align='center'\n    ),\n    cells=dict(\n        values=list(zip(*state_14_definitions)),\n        fill_color=['#2E7D32']*4 + ['#C62828']*4 + ['#F57F17']*4 + ['#1565C0']*2,\n        font=dict(color='white', size=12),\n        align='center',\n        height=35\n    )\n)])\n\nfig_states_14.update_layout(\n    title='<b>14种市场状态完整定义表</b>',\n    height=600,\n    paper_bgcolor='#1a1a2e',\n    font=dict(color='white')\n)\nfig_states_14.show()\n

项目路径: /home/taotao/dev/QuantTest/TRQuant
当前时间: 2026-01-03 14:27:44


In [23]:
import jqdatasdk as jq

with open(PROJECT_ROOT / 'config/jqdata_config.json') as f:
    jq_cfg = json.load(f)
jq.auth(jq_cfg['username'], jq_cfg['password'])
print(f"✅ JQData 认证成功, 剩余: {jq.get_query_count()['spare']}")

✅ JQData 认证成功, 剩余: 198522591


## 2️⃣ 导入核心模块

In [24]:
from core.market_env_identifier_v3 import (
    MarketEnvIdentifierV3, MarketEnvironment, identify_market_env_v3
)
from core.market_env_params_extended import (
    get_extended_params, get_params_from_result, format_params_summary
)
from core.visualization.chart_engine import ChartEngine
from core.visualization.dashboard import MarketGauge

import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("✅ 核心模块导入成功")

✅ 核心模块导入成功


## 3️⃣ 14种市场状态定义

系统将市场环境分为14种状态，分属4大类别：
- **牛市系列(4)**: 强势牛市、正常牛市、牛市后期、牛市回调
- **熊市系列(4)**: 强势熊市、正常熊市、熊市后期、熊市反弹
- **震荡系列(4)**: 高位震荡、中位震荡、低位震荡、宽幅震荡
- **转折系列(2)**: 底部反转、顶部反转


In [25]:
# 14种市场状态定义表
state_14_definitions = [
    ['牛市', '强势牛市', '全面上涨，远超年线，接近年内高点', '80%~100%', '低'],
    ['牛市', '正常牛市', '稳健上涨，高于年线，位于较高位置', '60%~80%', '低'],
    ['牛市', '牛市后期', '上涨动能减弱，位置较高', '40%~60%', '中'],
    ['牛市', '牛市回调', '中期趋势向上，短期回调', '50%~70%', '中'],
    ['熊市', '强势熊市', '全面下跌，远低于年线，接近年内低点', '0%~20%', '高'],
    ['熊市', '正常熊市', '稳定下跌，低于年线', '10%~30%', '高'],
    ['熊市', '熊市后期', '下跌动能减弱，位置较低', '30%~50%', '中'],
    ['熊市', '熊市反弹', '中期趋势向下，短期反弹', '20%~40%', '中'],
    ['震荡', '高位震荡', '在年内高位区间波动', '40%~60%', '中'],
    ['震荡', '中位震荡', '在年内中间位置波动', '30%~50%', '中'],
    ['震荡', '低位震荡', '在年内低位区间波动', '40%~60%', '中'],
    ['震荡', '宽幅震荡', '大幅波动但无明确方向', '30%~50%', '高'],
    ['转折', '底部反转', '长期下跌后开始上涨', '50%~70%', '中'],
    ['转折', '顶部反转', '长期上涨后开始下跌', '20%~40%', '高'],
]

fig_states = go.Figure(data=[go.Table(
    header=dict(
        values=['类别', '状态名称', '特征描述', '建议仓位', '风险等级'],
        fill_color='#2d2d44',
        font=dict(color='white', size=13),
        align='center'
    ),
    cells=dict(
        values=list(zip(*state_14_definitions)),
        fill_color=['#2E7D32']*4 + ['#C62828']*4 + ['#F57F17']*4 + ['#1565C0']*2,
        font=dict(color='white', size=12),
        align='center',
        height=35
    )
)])

fig_states.update_layout(
    title='<b>14种市场状态定义表</b>',
    height=600,
    paper_bgcolor='#1a1a2e',
    font=dict(color='white')
)
fig_states.show()


## 4️⃣ 配置分析参数

In [26]:
INDEX_CONFIG = {
    '000001.XSHG': {'name': '上证指数', 'color': '#F44336'},
    '399001.XSHE': {'name': '深证成指', 'color': '#2196F3'},
    '399006.XSHE': {'name': '创业板指', 'color': '#4CAF50'},
    '000688.XSHG': {'name': '科创50', 'color': '#FF9800'},
}

LOOKBACK_DAYS = 300
TODAY = datetime.now().strftime('%Y-%m-%d')

print(f"分析日期: {TODAY}")
print(f"分析指数: {', '.join([v['name'] for v in INDEX_CONFIG.values()])}")

分析日期: 2026-01-03
分析指数: 上证指数, 深证成指, 创业板指, 科创50


## 5️⃣ 获取市场数据

In [27]:
def get_index_data(symbol: str, count: int = 300) -> pd.DataFrame:
    return jq.get_price(symbol, count=count, end_date=TODAY, frequency='daily',
                        fields=['open', 'high', 'low', 'close', 'volume'])

data_dict = {}
for symbol, info in INDEX_CONFIG.items():
    try:
        df = get_index_data(symbol, LOOKBACK_DAYS)
        if df is not None and len(df) > 0:
            data_dict[symbol] = df
            print(f"✅ {info['name']}: {len(df)}条, 最新: {df.index[-1].strftime('%Y-%m-%d')}")
    except Exception as e:
        print(f"❌ {info['name']}: {e}")

print(f"\n共获取 {len(data_dict)} 个指数数据")

✅ 上证指数: 300条, 最新: 2025-12-31
✅ 深证成指: 300条, 最新: 2025-12-31
✅ 创业板指: 300条, 最新: 2025-12-31
✅ 科创50: 300条, 最新: 2025-12-31

共获取 4 个指数数据


## 6️⃣ 市场环境识别

**算法**: TrendAnalyzer (80%) + HMM (20%)

In [28]:
identifier = MarketEnvIdentifierV3()
env_results = {}

for symbol, df in data_dict.items():
    try:
        result = identifier.identify(df)  # 传入DataFrame
        env_results[symbol] = result
        name = INDEX_CONFIG[symbol]['name']
        print(f"✅ {name}: {result.combined_environment.value} (得分: {result.combined_score:.1f}, 置信度: {result.combined_confidence:.1%})")
    except Exception as e:
        print(f"❌ {INDEX_CONFIG[symbol]['name']}: {e}")

print(f"\n成功识别 {len(env_results)} 个指数")

✅ 上证指数: 牛市 (得分: 32.3, 置信度: 9378.6%)
✅ 深证成指: 牛市 (得分: 43.9, 置信度: 9969.6%)
✅ 创业板指: 牛市 (得分: 28.1, 置信度: 9539.6%)
✅ 科创50: 中位震荡 (得分: 6.0, 置信度: 3490.6%)

成功识别 4 个指数


## 7️⃣ 核心仪表盘

显示趋势强度、风险水平、建议仓位三大核心指标。

In [29]:
mg = MarketGauge()

if '000001.XSHG' in env_results:
    result = env_results['000001.XSHG']
    
    # 注意：参数名是score, risk_score, position，不是value
    fig_trend = mg.create_trend_gauge(score=result.combined_score, title="趋势强度")
    
    risk_map = {'low': 30, 'medium': 60, 'high': 90}
    risk_score = risk_map.get(result.risk_level, 50)
    fig_risk = mg.create_risk_gauge(risk_score=risk_score, title="风险水平")
    
    position_pct = (result.position_min + result.position_max) / 2
    fig_position = mg.create_position_gauge(position=position_pct, title="建议仓位")
    
    fig = make_subplots(
        rows=1, cols=3,
        specs=[[{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}]],
        subplot_titles=["趋势强度", "风险水平", "建议仓位"]
    )
    
    fig.add_trace(fig_trend.data[0], row=1, col=1)
    fig.add_trace(fig_risk.data[0], row=1, col=2)
    fig.add_trace(fig_position.data[0], row=1, col=3)
    
    fig.update_layout(
        title=f"<b>上证指数核心仪表盘</b> | 市场环境: {result.combined_environment.value}",
        height=350,
        paper_bgcolor='#1a1a2e',
        font=dict(color='white')
    )
    fig.show()
else:
    print("❌ 上证指数数据不可用")

## 8️⃣ 四大指数市场状态矩阵

In [30]:
def create_state_matrix(env_results: dict, index_config: dict) -> go.Figure:
    env_colors = {
        '强势牛市': '#00C853', '牛市': '#4CAF50', '弱势牛市': '#8BC34A',
        '强势熊市': '#F44336', '熊市': '#E91E63', '弱势熊市': '#FF5722',
        '高位震荡': '#FF9800', '中位震荡': '#FFC107', '低位震荡': '#FFEB3B',
        '复苏': '#2196F3', '派发': '#9C27B0', '中性': '#9E9E9E'
    }
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[index_config[s]['name'] for s in list(env_results.keys())[:4]],
        vertical_spacing=0.15, horizontal_spacing=0.08,
        specs=[[{"type": "table"}, {"type": "table"}], [{"type": "table"}, {"type": "table"}]]
    )
    
    for i, (symbol, result) in enumerate(list(env_results.items())[:4]):
        row, col = i // 2 + 1, i % 2 + 1
        env_name = result.combined_environment.value
        env_color = env_colors.get(env_name, '#9E9E9E')
        
        values = [
            ['综合环境', '周线状态', '月线状态', '季线状态', '综合得分', '置信度', '建议仓位'],
            [env_name, result.weekly.environment.value, result.monthly.environment.value,
             result.quarterly.environment.value, f"{result.combined_score:.1f}",
             f"{result.combined_confidence:.1%}", f"{result.position_min:.0%}~{result.position_max:.0%}"]
        ]
        
        fig.add_trace(
            go.Table(
                header=dict(values=['指标', '数值'], fill_color='#2d2d44', font=dict(color='white', size=12), align='center'),
                cells=dict(values=values, fill_color=[['#1a1a2e']*7, ['#1a1a2e']*6+[env_color]], font=dict(color='white', size=11), align='center')
            ), row=row, col=col
        )
    
    fig.update_layout(title='<b>A股四大指数市场状态矩阵</b>', height=600, paper_bgcolor='#1a1a2e', font=dict(color='white'))
    return fig

if len(env_results) >= 1:
    fig_matrix = create_state_matrix(env_results, INDEX_CONFIG)
    fig_matrix.show()

## 9️⃣ 三周期趋势预测矩阵

In [31]:
def create_period_matrix(env_results: dict, index_config: dict) -> go.Figure:
    indices = [index_config[s]['name'] for s in env_results.keys()]
    periods = ['周线', '月线', '季线']
    
    z_data, text_data = [], []
    for symbol, result in env_results.items():
        row_z, row_text = [], []
        for p in [result.weekly, result.monthly, result.quarterly]:
            row_z.append(p.score)
            row_text.append(f"{p.environment.value}\n{p.score:.1f}")
        z_data.append(row_z)
        text_data.append(row_text)
    
    fig = go.Figure(data=go.Heatmap(
        z=z_data, x=periods, y=indices, text=text_data, texttemplate="%{text}",
        textfont=dict(size=11, color='white'),
        colorscale=[[0, '#F44336'], [0.5, '#FFC107'], [1, '#00C853']],
        zmid=0, zmin=-100, zmax=100,
        colorbar=dict(title='得分', tickvals=[-100, -50, 0, 50, 100])
    ))
    
    fig.update_layout(
        title='<b>三周期趋势预测矩阵</b>', xaxis_title='周期', yaxis_title='指数',
        height=400, paper_bgcolor='#1a1a2e', plot_bgcolor='#1a1a2e', font=dict(color='white')
    )
    return fig

if env_results:
    fig_period = create_period_matrix(env_results, INDEX_CONFIG)
    fig_period.show()

## 🔟 多指数技术走势对比

In [32]:
ce = ChartEngine(backend='plotly')
named_data = {INDEX_CONFIG[s]['name']: df for s, df in data_dict.items()}

fig_price = ce.plot_multi_index_comparison(
    data_dict=named_data, chart_type='price', normalize=True,
    title='四大指数价格走势对比 (归一化=100)', height=600
)
if fig_price:
    fig_price.update_layout(paper_bgcolor='#1a1a2e', plot_bgcolor='#1a1a2e', font=dict(color='white'))
    fig_price.show()

In [33]:
fig_kline = ce.plot_multi_index_kline_grid(
    data_dict=named_data, ma_periods=[5, 20, 60],
    title='四大指数K线图 (MA5/20/60)', height=900
)
if fig_kline:
    fig_kline.update_layout(paper_bgcolor='#1a1a2e', plot_bgcolor='#1a1a2e', font=dict(color='white'))
    fig_kline.show()

In [34]:
fig_corr = ce.plot_multi_index_comparison(data_dict=named_data, chart_type='correlation', title='四大指数收益率相关性')
if fig_corr:
    fig_corr.update_layout(paper_bgcolor='#1a1a2e', plot_bgcolor='#1a1a2e', font=dict(color='white'))
    fig_corr.show()

## 1️⃣1️⃣ 下游操作参数表

In [35]:
if '000001.XSHG' in env_results:
    result = env_results['000001.XSHG']
    params = get_params_from_result(result)
    print(format_params_summary(params))

市场环境: 牛市 (bull)

【仓位管理】
  建议仓位: 70% (60% ~ 80%)
  单股上限: 12%
  最大持仓: 10只

【风险控制】
  风险等级: low
  止损线: 10%
  止盈线: 30%
  移动止损: 6%
  最大回撤: 15%

【调仓设置】
  调仓频率: weekly
  触发阈值: 8%

【买入条件】
  顺势而为，回调买入
  - 均线: 价格>MA20，MA20向上
  - RSI: (35, 75)

【卖出条件】
  目标止盈+移动止损结合
  - 止损类型: trailing
  - 止盈类型: target

【行业配置】
  偏好: 科技, 消费, 金融
  回避: 周期弱势行业

【操作建议】
  牛市阶段，保持较高仓位。顺势操作，回调加仓。关注领涨板块和龙头个股。

【关注指标】
  MA20趋势, 板块强度, 资金流向, 北向资金


In [36]:
def create_params_table(params) -> go.Figure:
    categories = ['仓位管理']*4 + ['风险控制']*5 + ['调仓设置']*2 + ['选股标准']*3
    items = ['最低仓位', '最高仓位', '建议仓位', '单股上限',
             '风险等级', '止损线', '止盈线', '移动止损', '最大回撤',
             '调仓频率', '触发阈值', '最大持仓', '最小市值', '流动性要求']
    values = [
        f"{params.position_min:.0%}", f"{params.position_max:.0%}",
        f"{params.suggested_position:.0%}", f"{params.single_stock_max:.0%}",
        params.risk_level, f"{params.stop_loss_pct:.0%}", f"{params.take_profit_pct:.0%}",
        f"{params.trailing_stop_pct:.0%}", f"{params.drawdown_limit:.0%}",
        params.rebalance_frequency, f"{params.rebalance_threshold:.0%}",
        f"{params.max_holdings}只", f"{params.min_market_cap}亿", params.liquidity_requirement
    ]
    
    fig = go.Figure(data=[go.Table(
        header=dict(values=['类别', '参数', '数值'], fill_color='#2d2d44', font=dict(color='white', size=13), align='center'),
        cells=dict(values=[categories, items, values], fill_color='#1a1a2e', font=dict(color='white', size=12), align='center', height=30)
    )])
    fig.update_layout(title=f'<b>下游操作参数 - {params.environment_name}</b>', height=550, paper_bgcolor='#1a1a2e', font=dict(color='white'))
    return fig

if '000001.XSHG' in env_results:
    fig_params = create_params_table(params)
    fig_params.show()

## 1️⃣2️⃣ 操作建议汇总

In [37]:
if '000001.XSHG' in env_results:
    print("=" * 60)
    print(f"📊 当前市场环境: {params.environment_name}")
    print("=" * 60)
    print(f"\n📋 操作建议:\n   {params.operation_advice}")
    print(f"\n🎯 关注指标: {', '.join(params.key_indicators)}")
    print(f"\n✅ 偏好行业: {', '.join(params.industry_preference)}")
    print(f"\n❌ 回避行业: {', '.join(params.industry_avoid)}")
    print("\n" + "=" * 60)

📊 当前市场环境: 牛市

📋 操作建议:
   牛市阶段，保持较高仓位。顺势操作，回调加仓。关注领涨板块和龙头个股。

🎯 关注指标: MA20趋势, 板块强度, 资金流向, 北向资金

✅ 偏好行业: 科技, 消费, 金融

❌ 回避行业: 周期弱势行业



## 1️⃣3️⃣ 交互式更新

In [38]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("⚠️ ipywidgets 未安装")

if WIDGETS_AVAILABLE and env_results:
    index_dropdown = widgets.Dropdown(
        options=[(INDEX_CONFIG[s]['name'], s) for s in env_results.keys()],
        value=list(env_results.keys())[0], description='选择指数:'
    )
    output = widgets.Output()
    
    def on_index_change(change):
        with output:
            clear_output(wait=True)
            symbol = change['new']
            if symbol in env_results:
                r = env_results[symbol]
                print(f"\n📊 {INDEX_CONFIG[symbol]['name']} 市场环境分析")
                print(f"   综合环境: {r.combined_environment.value}")
                print(f"   综合得分: {r.combined_score:.1f}")
                print(f"   置信度: {r.combined_confidence:.1%}")
                print(f"   建议仓位: {r.position_min:.0%} ~ {r.position_max:.0%}")
                print(f"   风险等级: {r.risk_level}")
                print(f"\n   周线: {r.weekly.environment.value} ({r.weekly.score:.1f})")
                print(f"   月线: {r.monthly.environment.value} ({r.monthly.score:.1f})")
                print(f"   季线: {r.quarterly.environment.value} ({r.quarterly.score:.1f})")
    
    index_dropdown.observe(on_index_change, names='value')
    display(index_dropdown)
    display(output)
    on_index_change({'new': list(env_results.keys())[0]})

Dropdown(description='选择指数:', options=(('上证指数', '000001.XSHG'), ('深证成指', '399001.XSHE'), ('创业板指', '399006.XSHE…

Output()

---## 📝 总结| 功能 | 说明 ||------|------|| 状态定义 | 14种市场状态分4大类别 || 核心算法 | TrendAnalyzer(80%) + HMM(20%) || 多指数 | 上证/深证/创业板/科创50 || 三周期 | 周/月/季独立判断 || 可视化 | ChartEngine + MarketGauge || 下游参数 | 仓位/风险/买卖条件 |